# use pretrained Resnet model to generate latent space, use Conv_autoencoder to get latent space of other sensors

data: only labeled data, umineko 2018 back 1

In [1]:
import os

import matplotlib.pyplot as plt
# import numpy as np
# import os
# import matplotlib.pyplot as plt
# import pickle
# import pandas as pd
# from torch.utils.data import Dataset, DataLoader
# import torch
# import torch.nn as nn
# 
# from scipy.interpolate import interp1d
# import math
# from datetime import datetime
# from tqdm import tqdm
# from sklearn.manifold import TSNE
# 
# import pickle
import torch.optim as optim
# from utils import *
# from hubconf import load_weights
# from sslearning.models.accNet import Resnet
from umap import UMAP
import plotly.express as px
# from importlib.metadata import version, PackageNotFoundError

import pickle
import numpy as np
from tqdm import tqdm
import torch

from deepview.calculate_results.data.umineko_data import (
    read_umineko_data,
read_umineko_path,
extract_data_from_year_back,
label_dict,
get_accel_batch_data,
get_gps_raw_batch_data,
get_gps_batch_data,

)

from deepview.calculate_results.models.utils import (
    Resnet,
load_weights,
MSEloss,
# torch,
AE_eval_time_series,
AE_train_time_series_resnet,
AE_train_time_series,
majority_value,
# np,
adjust_learning_rate,
# tqdm
Autoencoder3d,
Autoencoder1d,
Autoencoder2d,
)

from deepview.clustering_pytorch.datasets.factory import sliding_window

In [2]:
back_label_path = r'D:\logbot-data\BioTaggerData\masterLabelsByOtsuka\animal_id.csv'
umi_root_path = r'D:\logbot-data\BioTaggerData\export\raw\umineko\v.1.0.0'
file_paths = read_umineko_path(umi_root_path)

year = '2018'
data_path = r'D:\code\DeepView\deepview\calculate_results\data\umineko_%s.npy'%year
if os.path.exists(data_path):
    tmp = np.load(data_path, allow_pickle='TRUE').item()
    df_list = tmp['raw_data']
    df_clean_list = tmp['labeled_data']
    
selected_df, selected_clean_df = (
        extract_data_from_year_back(df_list, df_clean_list, 1))

selected_df['label_id'] = selected_df['label'].map(label_dict)
selected_clean_df['label_id'] = selected_clean_df['label'].map(label_dict)

# 将df_list没有标签的位置赋值-2，标记灰色，其他label正常标记。观察是否有label的标记能在不同灰色cluster中
# Replace NaN with -2 in column A
selected_df['label_id'] = selected_df['label_id'].fillna(-2)

# device = 'cuda'
# len_sw = 300
# sensor_type = 'GPS'
# train_loader = get_gps_batch_data(selected_df, 
#                                     len_sw, 
#                                     batch_size=512, 
#                                     device=device)

C:\Users\dell\AppData\Local\Temp\ipykernel_3484\3375479320.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_df['label_id'] = selected_df['label'].map(label_dict)
C:\Users\dell\AppData\Local\Temp\ipykernel_3484\3375479320.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_clean_df['label_id'] = selected_clean_df['label'].map(label_dict)
C:\Users\dell\AppData\Local\Temp\ipykernel_3484\3375479320.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Dat

In [3]:
device = 'cuda'
len_sw = 300
sensor_type = 'GPS'
selected_columns = ['GPS_velocity', 'GPS_bearing',
                        'label_id']  # without timestamps
df_selected = selected_df[selected_columns]
df_ffill = df_selected.fillna(method='bfill')
df_ffill = df_ffill.fillna(method='ffill')
tmp_b = sliding_window(df_ffill, len_sw)

# concatenate list
data_b = np.transpose(tmp_b[:, :, :-1], (0, 2, 1))  # [B, Len, dim-1] -> [B, dim-1, Len]
label_b = tmp_b[:, :, -1]  # [B, Len]


C:\Users\dell\AppData\Local\Temp\ipykernel_3484\1100558239.py:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_ffill = df_selected.fillna(method='bfill')
C:\Users\dell\AppData\Local\Temp\ipykernel_3484\1100558239.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_ffill = df_ffill.fillna(method='ffill')


In [4]:
# Initialize lists to store results for each row
averages = []
mins = []
maxs = []
medians = []
std_devs = []

# Iterate over each batch
for b in range(data_b.shape[0]):
    batch_averages = []
    batch_mins = []
    batch_maxs = []
    batch_medians = []
    batch_std_devs = []
    
    # Iterate over each row
    for row in range(data_b.shape[1]):
        line = data_b[b, row, :]
        
        # Calculate statistics
        avg = np.mean(line)
        min_val = np.min(line)
        max_val = np.max(line)
        median = np.median(line)
        std_dev = np.std(line)
        
        batch_averages.append(avg)
        batch_mins.append(min_val)
        batch_maxs.append(max_val)
        batch_medians.append(median)
        batch_std_devs.append(std_dev)
    
    averages.append(batch_averages)
    mins.append(batch_mins)
    maxs.append(batch_maxs)
    medians.append(batch_medians)
    std_devs.append(batch_std_devs)

# Convert lists to arrays for easier handling
averages = np.array(averages)
mins = np.array(mins)
maxs = np.array(maxs)
medians = np.array(medians)
std_devs = np.array(std_devs)

# combine features 
features = np.concatenate([averages, mins, maxs, medians, std_devs], axis=1)

In [12]:
avg

199.74971399477906

In [13]:
min_val

199.74971399477903

In [14]:
median

199.74971399477903

In [15]:
std_dev

2.842170943040401e-14

In [5]:
averages.shape

(56029, 2)

In [8]:
features = np.concatenate([averages, mins, maxs, medians, std_devs], axis=1)
features.shape

(56029, 10)

In [9]:
label_b.shape

(56029, 300)

In [11]:
label_concat_vote = majority_value(label_b)

umap_3d = UMAP(n_components=3)

proj_3d_gyro = umap_3d.fit_transform(features)

# set point size 
point_size = np.ones(proj_3d_gyro.shape[0]) * 1
grey_idx = np.where(label_concat_vote==-2)[0]
point_size[grey_idx] = 0.5

fig_3d = px.scatter_3d(
    proj_3d_gyro, x=0, y=1, z=2,
    color=label_concat_vote.astype(str),  
    labels={'color': 'activity'},
    # color_discrete_map={ '-2.0': ('rgba(239, 239, 240, 1)')},
    color_discrete_map={ '-2.0': 'grey'},
    size=point_size
)

# Update transparency for traces where activity is '-2.0'
fig_3d.for_each_trace(lambda trace: trace.update(marker=dict(opacity=0.5)) if trace.name == '-2.0' else ())

fig_3d.update_traces(marker=dict(line=dict(width=0)))  # remove boundary of point

fig_3d.show()

D:\Users\dell\anaconda3\envs\deepviewbackup\lib\site-packages\sklearn\manifold\_spectral_embedding.py:273: UserWarning:

Graph is not fully connected, spectral embedding may not work as expected.

D:\Users\dell\anaconda3\envs\deepviewbackup\lib\site-packages\sklearn\manifold\_spectral_embedding.py:392: UserWarning:

Exited at iteration 2000 with accuracies 
[3.40968762e-15 7.93647749e-07 1.33794351e-06 1.70657976e-06
 1.25915069e-05]
not reaching the requested tolerance 4.127621650695801e-06.
Use iteration 1536 instead with accuracy 
2.3312755276361064e-06.


D:\Users\dell\anaconda3\envs\deepviewbackup\lib\site-packages\sklearn\manifold\_spectral_embedding.py:392: UserWarning:

Exited postprocessing with accuracies 
[2.53242563e-15 8.48315114e-07 1.43313023e-06 1.79217924e-06
 7.58275312e-06]
not reaching the requested tolerance 4.127621650695801e-06.

D:\Users\dell\anaconda3\envs\deepviewbackup\lib\site-packages\umap\spectral.py:548: UserWarning:

Spectral initialisation failed! The e

In [ ]:
# only show the activities of -2 and 0

# find indices
value_idx = np.where((label_concat_vote==-2) | (label_concat_vote==0))[0]

fig_3d = px.scatter_3d(
    proj_3d_gyro[value_idx, :], x=0, y=1, z=2,
    color=label_concat_vote[value_idx].astype(str),  
    labels={'color': 'activity'},
    # color_discrete_map={ '-2.0': ('rgba(239, 239, 240, 1)')},
    color_discrete_map={ '-2.0': 'grey'},
    size=point_size[value_idx]
)

# Update transparency for traces where activity is '-2.0'
fig_3d.for_each_trace(lambda trace: trace.update(marker=dict(opacity=0.5)) if trace.name == '-2.0' else ())

fig_3d.update_traces(marker=dict(line=dict(width=0)))  # remove boundary of point

fig_3d.show()

In [ ]:
# only show the activities of -2 and 0

# find indices
value_idx = np.where((label_concat_vote==-2) | (label_concat_vote==14))[0]
# proj_3d_gyro[value_idx, :]
# label_concat_vote[value_idx]
# point_size[value_idx]

fig_3d = px.scatter_3d(
    proj_3d_gyro[value_idx, :], x=0, y=1, z=2,
    color=label_concat_vote[value_idx].astype(str),  
    labels={'color': 'activity'},
    # color_discrete_map={ '-2.0': ('rgba(239, 239, 240, 1)')},
    color_discrete_map={ '-2.0': 'grey'},
    size=point_size[value_idx]
)

# Update transparency for traces where activity is '-2.0'
fig_3d.for_each_trace(lambda trace: trace.update(marker=dict(opacity=0.5)) if trace.name == '-2.0' else ())

fig_3d.update_traces(marker=dict(line=dict(width=0)))  # remove boundary of point

fig_3d.show()

In [ ]:
# only show the activities of -2 and 0

# find indices
value_idx = np.where((label_concat_vote==-2) | (label_concat_vote==4))[0]
# proj_3d_gyro[value_idx, :]
# label_concat_vote[value_idx]
# point_size[value_idx]

fig_3d = px.scatter_3d(
    proj_3d_gyro[value_idx, :], x=0, y=1, z=2,
    color=label_concat_vote[value_idx].astype(str),  
    labels={'color': 'activity'},
    # color_discrete_map={ '-2.0': ('rgba(239, 239, 240, 1)')},
    color_discrete_map={ '-2.0': 'grey'},
    size=point_size[value_idx]
)

# Update transparency for traces where activity is '-2.0'
fig_3d.for_each_trace(lambda trace: trace.update(marker=dict(opacity=0.5)) if trace.name == '-2.0' else ())

fig_3d.update_traces(marker=dict(line=dict(width=0)))  # remove boundary of point

fig_3d.show()